In [ ]:
import torch
import torch.nn as nn
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.ops as ops

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

epochs = 25
batch_size = 100
learning_rate = 0.001

CIFAR100_MEAN = [0.5071, 0.4865, 0.4409]
CIFAR100_STD = [0.2673, 0.2564, 0.2762]
DATA_ROOT = './cifar100_data'

train_transforms = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD)
])

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD)
])

train_dataset = datasets.CIFAR100(root=DATA_ROOT, train=True, transform=train_transforms, download=True)
test_dataset = datasets.CIFAR100(root=DATA_ROOT, train=False, transform=test_transforms)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
def create_conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, groups=1):
    return nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride,
                     padding=padding, groups=groups)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, width, cardinality=32, expansion_rate=4, stride=1, downsample=None):
        super().__init__()
        mid_channels = cardinality * width
        self.downsample = downsample
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.conv1 = create_conv2d(in_channels, mid_channels, 1, stride)
        self.bn2 = nn.BatchNorm2d(mid_channels)
        self.conv2 = create_conv2d(mid_channels, mid_channels, 3, padding=1, groups=cardinality)
        self.bn3 = nn.BatchNorm2d(mid_channels)
        self.conv3 = create_conv2d(mid_channels, mid_channels * expansion_rate, 1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x

        out = self.bn1(x)
        out = self.relu(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out = self.conv1(out)

        out = self.bn2(out)
        out = self.relu(out)
        out = self.conv2(out)

        out = self.bn3(out)
        out = self.relu(out)
        out = self.conv3(out)

        out += residual

        return out

In [ ]:
class ResNeXt_50(nn.Module):
    def __init__(self, block, layers, num_classes=100):
        super().__init__()
        self.in_channels = 64
        # self.conv = create_conv2d(3, 64, 7, 2, 3)
        self.conv = create_conv2d(3, 64, 3, 1, 1)
        self.bn = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        # self.max_pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self.create_layer(block, 2, layers[0])
        self.layer2 = self.create_layer(block, 4, layers[1], stride=2)
        self.layer3 = self.create_layer(block, 8, layers[2], stride=2)
        self.layer4 = self.create_layer(block, 16, layers[3], stride=2)
        self.final_bn = nn.BatchNorm2d(2048)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(2048, num_classes)

    def create_layer(self, block, width, blocks, expansion_rate=4, cardinality=32, stride=1):
        mid_channels = cardinality * width
        out_channels = mid_channels * expansion_rate
        downsample = None
        if self.in_channels != out_channels or stride != 1:
            downsample = nn.Sequential(
                nn.Conv2d(in_channels=self.in_channels, out_channels=out_channels, kernel_size=1, stride=stride,
                          bias=False)
            )

        layers = []
        layers.append(block(in_channels=self.in_channels, width=width, expansion_rate=expansion_rate, stride=stride, downsample=downsample))
        self.in_channels = mid_channels * expansion_rate

        for i in range(1, blocks):
            layers.append(block(in_channels=self.in_channels, width=width, expansion_rate=expansion_rate))

        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv(x)
        out = self.bn(out)
        out = self.relu(out)
        # out = self.max_pool(out)

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out = self.final_bn(out)
        out = self.relu(out)

        out = self.avg_pool(out).view(out.size(0), -1)
        out = self.fc(out)

        return out

In [ ]:
model = ResNeXt_50(ResidualBlock, [3, 4, 6, 3]).to(device)
loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

model.train()
for epoch in range(epochs):

    for i, (inputs, labels) in enumerate(train_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        loss = loss_func(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i + 1) % 100 == 0:
            print("Epoch [{}/{}], Step [{}/{}] Loss: {:.4f}"
                  .format(epoch + 1, epochs, i + 1, len(train_loader), loss.item()))

    scheduler.step()

In [ ]:
model.eval()
with torch.no_grad():
    correct = 0
    total = 0

    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        (_, predicted) = torch.max(outputs.data, 1)
        correct += (predicted == labels).sum()

        loss = loss_func(outputs, labels)
        total += labels.size(0)

    print('Accuracy of the model on the test images: {} %'.format(100 * correct / total))